In [1]:
from mlflow import last_active_run, MlflowClient
#clean all variables, if needed
%reset -f

In [2]:
# define functions to get access token from Keycloak
import requests # 2.32.4
print("requests: ", requests.__version__)
import getpass

# Replace these with your Keycloak details
KEYCLOAK_URL = 'https://auth.simplex4learning.de/realms/simplex4learning/protocol/openid-connect/token'
CLIENT_ID = 'mlflow-api'

def get_access_token(username, password):
    payload = {
        'client_id': CLIENT_ID,
        'grant_type': 'password',
        'audience': 'mlflow',
        'scope': 'openid',
        'username': username,
        'password': password
    }

    response = requests.post(KEYCLOAK_URL, data=payload)

    if response.status_code == 200:
        token_data = response.json()
        access_token = token_data.get('access_token')
        expires_in = token_data.get('expires_in')
        return access_token, expires_in
    else:
        print(f"Failed to get access token: {response.status_code}")
        print(response.json())
        return None, None

def format_duration(seconds):
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    seconds = seconds % 60
    return f"{hours} hours, {minutes} minutes, {seconds} seconds"

requests:  2.32.5


In [3]:
# Running this part will prompt for username and password and save the access token
username = input("Enter your username: ")
password = getpass.getpass("Enter your password: ")

token, expires_in = get_access_token(username, password)

if token:
    print(f"Access Token: {token}")
    print(f"Token is valid for: {format_duration(expires_in)}")
else:
    print("Failed to retrieve access token.")

del expires_in
del username, password

Access Token: eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJqNUpOUlhIX0JWZUZQS1lRanFUZ3lyWi1rYVJoUmJybGtPOWlmZjdDSDJvIn0.eyJleHAiOjE3NTg1NTg5MjgsImlhdCI6MTc1ODUyMjkyOCwianRpIjoiNmNlZDQzNTUtY2MyNi00N2QwLWE2ODAtZmIxNDRlOTFjN2FmIiwiaXNzIjoiaHR0cHM6Ly9hdXRoLnNpbXBsZXg0bGVhcm5pbmcuZGUvcmVhbG1zL3NpbXBsZXg0bGVhcm5pbmciLCJhdWQiOlsibWxmbG93IiwiYWNjb3VudCJdLCJzdWIiOiJjN2U2YTU2NC0xZDAyLTQxZGEtOWRlOC0xNTA2ZTc0YzM5YjUiLCJ0eXAiOiJCZWFyZXIiLCJhenAiOiJtbGZsb3ctYXBpIiwic2Vzc2lvbl9zdGF0ZSI6IjUxZjM1ZDRkLThlY2YtNDFkMi04OTQ5LTM2ZTQ0ZjhiNDYzZSIsImFjciI6IjEiLCJhbGxvd2VkLW9yaWdpbnMiOlsiLyoiXSwicmVhbG1fYWNjZXNzIjp7InJvbGVzIjpbIm9mZmxpbmVfYWNjZXNzIiwidW1hX2F1dGhvcml6YXRpb24iLCJkZWZhdWx0LXJvbGVzLXNpbXBsZXg0bGVhcm5pbmciXX0sInJlc291cmNlX2FjY2VzcyI6eyJhY2NvdW50Ijp7InJvbGVzIjpbIm1hbmFnZS1hY2NvdW50IiwibWFuYWdlLWFjY291bnQtbGlua3MiLCJ2aWV3LXByb2ZpbGUiXX19LCJzY29wZSI6Im9wZW5pZCBwcm9maWxlIGVtYWlsIiwic2lkIjoiNTFmMzVkNGQtOGVjZi00MWQyLTg5NDktMzZlNDRmOGI0NjNlIiwiZW1haWxfdmVyaWZpZWQiOnRydWUsIm5hbWUiOiJNYXJpdXMgSGVycm1hbm

In [4]:
# Save the token to an environment variable for MLFlow
import os

os.environ["MLFLOW_TRACKING_TOKEN"] = token
os.environ["MLFLOW_TRACKING_URI"] = "https://mlflow.simplex4learning.de"
print("MLFlow Tracking URI and Token saved to environment variables and can be accessed there.")

del token

MLFlow Tracking URI and Token saved to environment variables and can be accessed there.


## Create a simple model and log it to MLFlow
Note: You need to get an access token first, see above.

In [71]:
import dask.dataframe as dd
import numpy as np
file_path = '../Data/water_temps.parquet'
rd = dd.read_parquet(file_path, npartitions = 4)
rd= rd.compute()
rd

,Jahr,Monat,Messwert (Maximum)
0,2007,5,"21,50"
1,2007,6,"23,80"
2,2007,7,"24,50"
3,2007,8,"22,90"
4,2007,9,"18,80"
...,...,...,...
113,2016,10,"18,40"
114,2016,11,"15,20"
115,2016,12,"9,20"
116,2017,1,"7,90"


In [72]:
rd.at[117,"Messwert (Maximum)"] = "15,30"

In [73]:
rd.tail()

,Jahr,Monat,Messwert (Maximum)
113,2016,10,"18,40"
114,2016,11,"15,20"
115,2016,12,"9,20"
116,2017,1,"7,90"
117,2017,2,"15,30"


In [74]:
import mlflow # 3.1.1
print("mlflow: ", mlflow.__version__)
from mlflow.models import infer_signature
import pandas as pd # 2.3.1
print("pandas: ", pd.__version__)
import sklearn # 1.7.1
print("sklearn: ", sklearn.__version__)

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, root_mean_squared_error

target = "Messwert (Maximum)"

X = rd.drop(target, axis=1)
temp_list = [item[0].replace(',', '.') for item in rd[[target]].values]
y = np.array(temp_list, dtype=float)

print("Splitting the dataset into training and test sets...")
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=17
)

# Ensure model is trained with feature names
feature_names = rd.columns.drop(target)
X_train = pd.DataFrame(X_train, columns=feature_names)
X_test = pd.DataFrame(X_test, columns=feature_names)
y_train = pd.Series(y_train, name="target")
y_test = pd.Series(y_test, name="target")

del feature_names
del X, y

print("Defining the model hyperparameters...")
# Define the model hyperparameters
params = {
    "criterion": "squared_error",
    "random_state": 17,
}

print("Training the model...")
# Train the model
lr = DecisionTreeRegressor(**params)
lr.fit(X_train, y_train)

print("Predicting on the test set...")
# Predict on the test set
y_pred = lr.predict(X_test)

print("Calculating metrics...")
# Calculate metric
mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

print("Training complete!")

mlflow:  3.3.2
pandas:  2.2.3
sklearn:  1.7.1
Splitting the dataset into training and test sets...
Defining the model hyperparameters...
Training the model...
Predicting on the test set...
Calculating metrics...
Training complete!


In [6]:
# Create a new experiment with given name.
UNIQUE_EXPERIMENT_NAME = "test-Gew"

print("Creating a new experiment with name: ", UNIQUE_EXPERIMENT_NAME, " in MLFlow...")
mlflow.set_experiment(experiment_name=UNIQUE_EXPERIMENT_NAME)

Creating a new experiment with name:  test-Gew  in MLFlow...


<Experiment: artifact_location='mlflow-artifacts:/10', creation_time=1757067181435, experiment_id='10', last_update_time=1757067181435, lifecycle_stage='active', name='test-Gew', tags={}>

In [76]:
# Read current conda environment file to save it with the model.
import os
absolutePathToProjectFolder = os.getcwd()
print('Project is located in: ' + absolutePathToProjectFolder + '')

conda_env_filePath = absolutePathToProjectFolder + '/spx_env.yml'
with open(conda_env_filePath, "r", encoding="utf-8", errors="replace") as f:
    conda_env = f.read()

print(conda_env)

del conda_env, absolutePathToProjectFolder, f

Project is located in: c:\Users\herrmann\git\simplex\Notebooks
name: spx
channels:
  - conda-forge
dependencies:
  - _openmp_mutex=4.5=2_gnu
  - aiohappyeyeballs=2.6.1=pyhd8ed1ab_0
  - aiohttp=3.12.15=py313hd650c13_0
  - aiosignal=1.4.0=pyhd8ed1ab_0
  - alembic=1.16.5=pyhd8ed1ab_0
  - anyio=4.10.0=pyhe01879c_0
  - appdirs=1.4.4=pyhd8ed1ab_1
  - argon2-cffi=25.1.0=pyhd8ed1ab_0
  - argon2-cffi-bindings=25.1.0=py313h5ea7bf4_0
  - arrow=1.3.0=pyhd8ed1ab_1
  - asttokens=3.0.0=pyhd8ed1ab_1
  - async-lru=2.0.5=pyh29332c3_0
  - attrs=25.3.0=pyh71513ae_0
  - aws-c-auth=0.9.0=hd9a66b3_19
  - aws-c-cal=0.9.2=hef2a5b8_1
  - aws-c-common=0.12.4=hfd05255_0
  - aws-c-compression=0.3.1=ha8a2810_6
  - aws-c-event-stream=0.5.5=hccb7587_3
  - aws-c-http=0.10.4=h04b3cea_0
  - aws-c-io=0.21.2=h20b9e97_1
  - aws-c-mqtt=0.13.3=h6b158f5_3
  - aws-c-s3=0.8.6=h46905be_2
  - aws-c-sdkutils=0.2.4=ha8a2810_1
  - aws-checksums=0.2.7=ha8a2810_2
  - aws-crt-cpp=0.33.1=h89ba1a2_2
  - aws-sdk-cpp=1.11.606=h14334ec_1
  

In [77]:
print("Logging model parameters, metrics, and artifacts to MLFlow...")
print("Everytime running this cell, a new run (unique) will be created.")

with mlflow.start_run() as run:

    # Log model parameters
    #mlflow.log_params(params=params)

    # Log metrics
    mlflow.log_metrics(
        metrics={"mean squared error": mse,
                 "root mean squared error": rmse
                }
    )

    # Log the model, which inherits the parameters and metric
    model_info = mlflow.sklearn.log_model(
        sk_model=lr,
        conda_env=conda_env_filePath,
        registered_model_name="Gewaesser Test",
        signature=infer_signature(X_train, y_pred),
        input_example=X_train[:10],
        metadata={
            "author": "marius",
            "version": "0.0.1",
        },
        params=params,
        tags={"Training Info": "Decision tree for water parameter measurement",
              "Additional Info": "Predictors are month and year, predictand is a measurement.",
              "Extra Info": "This model is not very accurate."
              },
        name="gew_model",
    )

    subfolder = "data_folder"

    mlflow.log_artifact(local_path=file_path,
                        artifact_path=subfolder,
                        )
    
    # get the run name and id
    run_name = run.info.run_name
    run_id = run.info.run_id

    print("\nRun name: ", run_name)
    print("Run id: ", run_id)

    logged_run = mlflow.get_run(run_id)

    artifact_url_parts = run.info.artifact_uri.split('/')
    experiment_num = artifact_url_parts[1]
    artifact_path = artifact_url_parts[2]

    dataset_source_url = os.path.join("https://mlflow.simplex4learning.de/#/experiments/", experiment_num, "runs", artifact_path, "artifacts", subfolder, file_path.split("/")[-1]).replace("\\","/")

    training_dataset = mlflow.data.from_pandas(rd, source = dataset_source_url, name = "maximum water temperatures", targets = "Messwert (Maximum)")
    mlflow.log_input(training_dataset, context="training")

    print("Complete: Everything is logged to MLFlow Server. Let's delete the variables now...")

mlflow.end_run()

# del mse, rmse, lr, params
# del X_train, y_pred, y_train
# del run_name, run_id, model_info, run, conda_env_filePath
# del subfolder


Logging model parameters, metrics, and artifacts to MLFlow...
Everytime running this cell, a new run (unique) will be created.


c:\Users\herrmann\AppData\Local\miniforge3\envs\spx\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/09/05 12:20:23 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - numpy (current: 2.2.0, required: numpy==2.3.2)
 - pan

2025/09/05 12:20:23 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - numpy (current: 2.2.0, required: numpy==2.3.2)
 - pandas (current: 2.2.3, required: pandas==2.3.2)
 - pytz (current: 2024.1, required: pytz==2025.2)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
Registered model 'Gewaesser Test' already exists. Creating a new version of this model...
2025/09/05 12:20:28 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Gewaesser Test, version 7
Created version '7' of model 'Gewaesser Test'.



Run name:  persistent-bass-645
Run id:  3cdd0c9d090c444fa8be1a4b7ac39882


c:\Users\herrmann\AppData\Local\miniforge3\envs\spx\Lib\site-packages\mlflow\data\dataset_source_registry.py:149: UserWarning: Failed to determine whether UCVolumeDatasetSource can resolve source information for 'https://mlflow.simplex4learning.de/#/experiments/10/runs/3cdd0c9d090c444fa8be1a4b7ac39882/artifacts/data_folder/water_temps.parquet'. Exception: 
  return _dataset_source_registry.resolve(
c:\Users\herrmann\AppData\Local\miniforge3\envs\spx\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have

Complete: Everything is logged to MLFlow Server. Let's delete the variables now...
🏃 View run persistent-bass-645 at: https://mlflow.simplex4learning.de/#/experiments/10/runs/3cdd0c9d090c444fa8be1a4b7ac39882
🧪 View experiment at: https://mlflow.simplex4learning.de/#/experiments/10


In [ ]:
                            # Debugging, if some run was not ended properly
#mlflow.end_run()

# Load previously logged model from MLFlow

In [5]:
import mlflow
print("Loading all available experiments from MLFlow Server...")
experiments = experiments = mlflow.search_experiments()
print("Number of experiments found: ", len(experiments))

import datetime

for experiment in experiments:
    print(experiment.experiment_id, "-",
          experiment.name,
          "(",datetime.datetime.fromtimestamp(experiment.creation_time/1000, tz=datetime.timezone.utc).strftime('%Y-%m-%d %H:%M:%S'),")",
          experiment.artifact_location
          )

del experiments, experiment


print("\nTry to load experiment with name: ", UNIQUE_EXPERIMENT_NAME, " from MLFlow Server...")
loaded_experiment = mlflow.get_experiment_by_name(UNIQUE_EXPERIMENT_NAME)

print("\nloaded experiment name: ", loaded_experiment.name)
print("loaded experiment with id: ", loaded_experiment.experiment_id)
print("")


print ("Loading all runs of the experiment...")
list_of_runs = mlflow.search_runs(loaded_experiment.experiment_id)
print("Number of runs found: ", len(list_of_runs))

for run in list_of_runs.iterrows():
    # run_info = run[1]
    # test = run_info.run_name
    print(run[1].run_id)

del run, loaded_experiment


# Now query meta data for the latest run
last_run_info = list_of_runs.iloc[0] # the first entry is the latest run
last_run_id = last_run_info.run_id

print("\nLoading attributes of the last run from MLFlow Server...")

print("\nAvailable attributes:")
for attr in last_run_info.index:
    print(f"{attr}: {last_run_info[attr]}")

del attr

run_attributes = last_run_info.to_dict()
run_belongs_to_experiment_number = run_attributes["experiment_id"]
run_id = run_attributes["run_id"]
run_artifact_uri = run_attributes["artifact_uri"]


# We only need the run_id! All other variables are just for additional information.
del run_attributes, run_artifact_uri, last_run_info, run_id


Loading all available experiments from MLFlow Server...
Number of experiments found:  8
10 - test-Gew ( 2025-09-05 10:13:01 ) mlflow-artifacts:/10
8 - Minhs Test Experiment ( 2025-09-03 13:27:56 ) mlflow-artifacts:/8
5 - Test Experiment ( 2025-08-21 07:58:30 ) mlflow-artifacts:/5
4 - lfb-insect-plague_pycaret_presentation ( 2025-04-04 08:11:54 ) mlflow-artifacts:/4
3 - lfb-insect-plague_pycaret_example ( 2025-04-01 09:24:31 ) mlflow-artifacts:/3
2 - lfb-insect-plague_pycaret ( 2025-03-24 10:20:45 ) mlflow-artifacts:/2
1 - diabetes_regression ( 2024-09-17 08:37:29 ) mlflow-artifacts:/1
0 - Default ( 2024-09-09 16:05:45 ) mlflow-artifacts:/0


NameError: name 'UNIQUE_EXPERIMENT_NAME' is not defined

In [7]:
os.environ["RUN_ID"] = "3cdd0c9d090c444fa8be1a4b7ac39882"

In [10]:
import mlflow.sklearn
from sklearn.datasets import make_regression

In [9]:
logged_run = mlflow.get_run(os.getenv('RUN_ID'))
logged_run

<Run: data=<RunData: metrics={'mean squared error': 125.49499999999999,
 'root mean squared error': 11.202455088059939}, params={}, tags={'mlflow.runName': 'persistent-bass-645',
 'mlflow.source.name': 'c:\\Users\\herrmann\\AppData\\Local\\miniforge3\\envs\\spx\\Lib\\site-packages\\ipykernel_launcher.py',
 'mlflow.source.type': 'LOCAL',
 'mlflow.user': 'herrmann'}>, info=<RunInfo: artifact_uri='mlflow-artifacts:/10/3cdd0c9d090c444fa8be1a4b7ac39882/artifacts', end_time=1757067631543, experiment_id='10', lifecycle_stage='active', run_id='3cdd0c9d090c444fa8be1a4b7ac39882', run_name='persistent-bass-645', start_time=1757067618616, status='FINISHED', user_id='herrmann'>, inputs=<RunInputs: dataset_inputs=[<DatasetInput: dataset=<Dataset: digest='e84aca90', name='maximum water temperatures', profile='{"num_rows": 118, "num_elements": 354}', schema=('{"mlflow_colspec": [{"type": "long", "name": "Jahr", "required": true}, '
 '{"type": "long", "name": "Monat", "required": true}, {"type": "strin

In [53]:
logged_run.outputs.to_dictionary()["model_outputs"][0].model_id

'm-0aaac473c259469ba5404fb34fa31370'

In [ ]:
artifact_path = logged_run.info.artifact_uri.split("/"+logged_run.info.run_id+"/")

In [69]:
model_path = os.path.join(artifact_path[0], "models", logged_run.outputs.to_dictionary()["model_outputs"][0].model_id, artifact_path[1], "model.pkl").replace("\\","/")
model_path

'mlflow-artifacts:/10/models/m-0aaac473c259469ba5404fb34fa31370/artifacts/model.pkl'

In [72]:
model = mlflow.artifacts.download_artifacts(model_path, dst_path = "./")


In [74]:
model

'c:\\Users\\herrmann\\git\\simplex\\Notebooks\\model.pkl'

In [75]:
import pickle

In [76]:
loaded_model = pickle.load(open(model, 'rb'))

In [77]:
loaded_model

,criterion,'squared_error'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,17
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [12]:
model_uri = "runs:/{}/model".format(logged_run.info.run_id)
loaded_model = mlflow.sklearn.load_model(model_uri)

MlflowException: Failed to download artifacts from path 'model', please ensure that the path is correct.

In [46]:
experiment = [str(10)]
list_of_runs = mlflow.search_runs(experiment)

print(list_of_runs)

                             run_id experiment_id    status  \
0  3cdd0c9d090c444fa8be1a4b7ac39882            10  FINISHED   
1  737fb78be5454cc9adf2706441d445c2            10  FINISHED   

                                        artifact_uri  \
0  mlflow-artifacts:/10/3cdd0c9d090c444fa8be1a4b7...   
1  mlflow-artifacts:/10/737fb78be5454cc9adf270644...   

                        start_time                         end_time  \
0 2025-09-05 10:20:18.616000+00:00 2025-09-05 10:20:31.543000+00:00   
1 2025-09-05 10:13:08.564000+00:00 2025-09-05 10:13:27.291000+00:00   

   metrics.root mean squared error  metrics.mean squared error  \
0                        11.202455                  125.495000   
1                        11.206149                  125.577778   

   tags.mlflow.runName tags.mlflow.user  \
0  persistent-bass-645         herrmann   
1   resilient-shad-998         herrmann   

                             tags.mlflow.source.name tags.mlflow.source.type  
0  c:\Users\herrman

In [54]:
import numpy as np

In [55]:
np.arange(0,10,1)

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [67]:
run_ids, run_names = [list_of_runs.iloc[i]["run_id"] for i in np.arange(0,len(list_of_runs),1)], [list_of_runs.iloc[i]["tags.mlflow.runName"] for i in np.arange(0,len(list_of_runs),1)]

In [68]:
logged_run = mlflow.get_run(run_ids[0])

In [69]:
logged_run

<Run: data=<RunData: metrics={'mean squared error': 125.49499999999999,
 'root mean squared error': 11.202455088059939}, params={}, tags={'mlflow.runName': 'persistent-bass-645',
 'mlflow.source.name': 'c:\\Users\\herrmann\\AppData\\Local\\miniforge3\\envs\\spx\\Lib\\site-packages\\ipykernel_launcher.py',
 'mlflow.source.type': 'LOCAL',
 'mlflow.user': 'herrmann'}>, info=<RunInfo: artifact_uri='mlflow-artifacts:/10/3cdd0c9d090c444fa8be1a4b7ac39882/artifacts', end_time=1757067631543, experiment_id='10', lifecycle_stage='active', run_id='3cdd0c9d090c444fa8be1a4b7ac39882', run_name='persistent-bass-645', start_time=1757067618616, status='FINISHED', user_id='herrmann'>, inputs=<RunInputs: dataset_inputs=[<DatasetInput: dataset=<Dataset: digest='e84aca90', name='maximum water temperatures', profile='{"num_rows": 118, "num_elements": 354}', schema=('{"mlflow_colspec": [{"type": "long", "name": "Jahr", "required": true}, '
 '{"type": "long", "name": "Monat", "required": true}, {"type": "strin

In [74]:
logged_run.inputs.dataset_inputs[0].dataset.source

'{"url": "https://mlflow.simplex4learning.de/#/experiments/10/runs/3cdd0c9d090c444fa8be1a4b7ac39882/artifacts/data_folder/water_temps.parquet"}'

In [80]:
subfolder = "data_folder"

In [92]:
artifact_url_parts = logged_run.info.artifact_uri.split('/')

In [93]:
experiment_num = artifact_url_parts[1]
artifact_path = artifact_url_parts[2]

In [ ]:
logged_run.info.artifact_uri

'mlflow-artifacts:/10/3cdd0c9d090c444fa8be1a4b7ac39882/artifacts'

In [85]:
logged_dataset = logged_run.inputs.dataset_inputs[0].dataset
logged_dataset

<Dataset: digest='e84aca90', name='maximum water temperatures', profile='{"num_rows": 118, "num_elements": 354}', schema=('{"mlflow_colspec": [{"type": "long", "name": "Jahr", "required": true}, '
 '{"type": "long", "name": "Monat", "required": true}, {"type": "string", '
 '"name": "Messwert (Maximum)", "required": true}]}'), source=('{"url": '
 '"https://mlflow.simplex4learning.de/#/experiments/10/runs/3cdd0c9d090c444fa8be1a4b7ac39882/artifacts/data_folder/water_temps.parquet"}'), source_type='http'>

In [102]:
# Loading the dataset's source
dataset_source = mlflow.data.get_source(logged_dataset)

# https://mlflow.org/docs/2.21.3/dataset/ 
# If not implemented, dataset was commited without source before (https://github.com/mlflow/mlflow/issues/13015)
# -> Need to slightly change dataset to change digest

# local_dataset = dataset_source.load()
# The solution above is nicer, but returns the authentication page
# While this works due to the data folder
training_data = mlflow.artifacts.download_artifacts(os.path.join(logged_run.info.artifact_uri, subfolder, file_path.split("/")[-1]).replace("\\","/"), dst_path = "./")

# print(f"The local file where the data has been downloaded to: {local_dataset}")

# Load the data again
loaded_data = dd.read_parquet(training_data)

In [104]:
loaded_data.compute()

,Jahr,Monat,Messwert (Maximum)
0,2007,5,"21,50"
1,2007,6,"23,80"
2,2007,7,"24,50"
3,2007,8,"22,90"
4,2007,9,"18,80"
...,...,...,...
113,2016,10,"18,40"
114,2016,11,"15,20"
115,2016,12,"9,20"
116,2017,1,"7,90"


# Save run data into a mocked database
- The database is mocked using a plain old csv file.

In [ ]:
def list_all_artifacts(run_id: str, root_path: str = ""):
    """
    Returns a list of all artifact file paths (relative to the run's artifact root)
    for the given run_id. Set root_path to start from a subdirectory if needed.
    """

    from mlflow import MlflowClient

    client = MlflowClient()
    all_files = []
    stack = [root_path]

    while stack:
        current = stack.pop()
        for fi in client.list_artifacts(run_id=run_id, path=current):
            if fi.is_dir:
                stack.append(fi.path)  # dive into subdirectory
            else:
                all_files.append(fi.path)  # collect file path
    return all_files

all_artifacts = list_all_artifacts(last_run_id)

print("All artifacts found for run with id: ", last_run_id)

if len(all_artifacts) == 0:
    print("No additional artifacts found. \n\nNote: There are sill the default artifacts available like: MLmodel, conda.yaml, input_example.json, model.pkl, python_env.yaml, requirements.txt, serving_input_example.json")
else:
    for artifact in all_artifacts:
        print(artifact)

#del all_artifacts

# Add extra metrics to the current run

In [ ]:
print("Adding extra metrics to the current run with id: ", last_run_id)

# Resume the existing run to append extra metrics
with mlflow.start_run(run_id=last_run_id) as run:
    #add some metrics
    mlflow.log_metric("test_ABC", 1)
    mlflow.log_metric("test_CDE", 2)

    #client = MlflowClient()

    run_data = mlflow.get_run(run_id=last_run_id)
    run_data.to_dictionary()

    # print out all attributes in run_data
    for attr in run_data.data.metrics.keys():
        print(f"{attr}: {run_data.data.metrics[attr]}")

    del attr, run_data

mlflow.end_run()

del run


In [ ]:
print("Saving run data and model data into a mocked database...")

# HERE WE ARE SIMULATING A DATABASE, so we can load the runs and models
# from different jupyter notebooks without using mlflow.
import os
absolutePathToProjectFolder = os.getcwd()
print('\nProject is located in: ' + absolutePathToProjectFolder + '')
local_data_path = absolutePathToProjectFolder + '/iris_example'
os.makedirs(local_data_path, exist_ok=True)
filePath_model_meta_db = local_data_path + '/model_meta_db.csv'
print('Metadata for all runs of: ' + UNIQUE_EXPERIMENT_NAME + ' will be saved to: ' + filePath_model_meta_db)
list_of_runs.to_csv(filePath_model_meta_db, index=False)

del local_data_path
del list_of_runs

# Add extra artifacts to the current run

In [75]:
last_run_id= logged_run.info.run_id

In [ ]:
print("Adding extra artifacts to the current run with id: ", last_run_id)

In [76]:
def make_test_plot(abs_folder_path: str = ""):
    """
    Creates a test plot and saves it to the given folder path.
    Returns the absolute file path of the saved plot.
    """
    import matplotlib.pyplot as plt
    import numpy as np

    # create a test plot for uploading this as a test artifact
    x = np.linspace(0, 10, 200)
    y = np.sin(x)
    plt.figure()
    plt.plot(x, y, label="sin(x)")
    plt.title("A Test Plot with sin(x)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    abs_file_path = os.path.join(abs_folder_path, "line.png")
    plt.savefig(abs_file_path)
    print("Saved test plot to: ", abs_file_path)
    print()
    plt.close()

    return abs_file_path

################################################################################
print("Adding extra artifacts to the current run with id: ", last_run_id)

import os
absolutePathToProjectFolder = os.getcwd()
print('Project is located in: ' + absolutePathToProjectFolder + '')

# create a local folder for artifacts
local_artifacts_path = absolutePathToProjectFolder + '/iris_example/artifacts_' + last_run_id
os.makedirs(local_artifacts_path, exist_ok=True)

print('\nSaving a test artifact to: ' + local_artifacts_path)
artifact_filePath = make_test_plot(local_artifacts_path)

# NOTE: When adding additional artifacts, all of the default artifacts
# (MLmodel, conda.yaml, input_example.json, model.pkl, python_env.yaml, requirements.txt, serving_input_example.json)
# will be moved to the logged modells of the run.
with mlflow.start_run(run_id=last_run_id) as run:

    subfolder = "extra_artifacts"

    # mlflow.log_artifact(local_path=artifact_filePath,
    #                     artifact_path=subfolder,
    #                     )

    mlflow.log_artifacts(local_dir=local_artifacts_path,
                         artifact_path=subfolder,
                         )
    del subfolder

mlflow.end_run()

del local_artifacts_path, artifact_filePath, absolutePathToProjectFolder,run


Adding extra artifacts to the current run with id:  3cdd0c9d090c444fa8be1a4b7ac39882
Project is located in: c:\Users\herrmann\git\simplex\Notebooks

Saving a test artifact to: c:\Users\herrmann\git\simplex\Notebooks/iris_example/artifacts_3cdd0c9d090c444fa8be1a4b7ac39882
Saved test plot to:  c:\Users\herrmann\git\simplex\Notebooks/iris_example/artifacts_3cdd0c9d090c444fa8be1a4b7ac39882\line.png

🏃 View run persistent-bass-645 at: https://mlflow.simplex4learning.de/#/experiments/10/runs/3cdd0c9d090c444fa8be1a4b7ac39882
🧪 View experiment at: https://mlflow.simplex4learning.de/#/experiments/10


# Loading default artifacts from the logged model
*Note: The logged Model != the Run*

In [ ]:
print("Loading default artifacts from the logged model with id: ", last_run_id)

In [ ]:
# Load the default artifacts from the logged model, which belong to the last run
print("Query all logged models from run: ", last_run_id)

experiments_list = [run_belongs_to_experiment_number]
# Error: MlflowException: experiment_ids must be a list of strings, got <class 'list'>

models_df = mlflow.search_logged_models(
    experiment_ids=experiments_list,
)

print("Number of logged models found for experiments: ", experiments_list)

del experiments_list


# Query dataframe for the logged model with the last run_id
model_df = models_df[models_df["source_run_id"] == last_run_id]

artifact_uri = model_df["artifact_location"].iloc[0]
model_id = model_df["model_id"].iloc[0]
model_name = model_df["name"].iloc[0]
model_params = model_df["params"].iloc[0]
model_status = model_df["status"].iloc[0]
model_tags = model_df["tags"].iloc[0]

print("Model name: ", model_name)
print("Model id: ", model_id)
print("Model params: ", model_params)
print("Model status: ", model_status)
print("Tags: ", model_tags)

########del models_df, model_df, model_name, model_params, model_status, model_tags

# get artifacts from the logged model
print("Loading artifacts from logged model with id: ", model_id)

model_artifacts = mlflow.artifacts.list_artifacts(artifact_uri=artifact_uri)

for artifact in model_artifacts:
    print(artifact)

print("")

#find model of artifact with path: 'artifacts/model.pkl' in model_artifacts
artifact_model_pkl = None
for artifact in model_artifacts:
    if artifact.path == 'artifacts/model.pkl':
            artifact_model_pkl = artifact
            break

del artifact

if artifact_model_pkl.file_size > 100000000: # 10 MB
    # print WARNING!!!
    print("\nWARNING!!! The size of the model.pkl artifact is larger than 10 MB\n")


# create a local folder for artifacts
import os
absolutePathToProjectFolder = os.getcwd()
local_artifacts_path = absolutePathToProjectFolder + '/iris_example/artifacts/' + model_id
os.makedirs(local_artifacts_path, exist_ok=True)


print("Artifacts from logged model with id: ", model_id)
print("will be saved to: ", local_artifacts_path)
print("\nIF DOWNLOADING TAKES TOO LONG, consider downloading the artifacts manually from the MLFlow Server.\n")
mlflow.artifacts.download_artifacts(artifact_uri=artifact_uri, dst_path=local_artifacts_path)

del absolutePathToProjectFolder,artifact_model_pkl, artifact_uri

In [98]:
import pandas as pd
import numpy as np
N1 = 100
N2 = 50

df1 = pd.DataFrame(np.random.rand(N1, 2))
df2 = pd.DataFrame(np.random.rand(N2, 2))

In [99]:
frac = df1.shape[0]/df2.shape[0]

In [104]:
if frac < 1: 
    df2 = df2.sample(frac=frac)
else:
    df1= df1.sample(frac=1/frac)

SyntaxError: invalid syntax. Maybe you meant '==' or ':=' instead of '='? (843482049.py, line 1)

In [150]:
c = ["a", "b", "c"]
d = ["a", "e", "c"]
e = ["Alex", "Bertha", "Carlos"]
g = ["Alex", "Gunther", "Carlos"]

In [156]:
f = set(e).intersection(g)
f = list(f)
f

['Carlos', 'Alex']

In [ ]:
rename_dict_new = dict(zip(c, e))
rename_dict_new.values()

dict_values(['Alex', 'Bertha', 'Carlos'])

In [192]:
g = {key:value for key, value in rename_dict_new.items() if value in f}
g

{'a': 'Alex', 'c': 'Carlos'}

In [170]:
for key, value in rename_dict_new.items():
    if value in f:
        print(key)

a
c


In [ ]:
for value in f:
    print()

False
False


In [158]:
rename_dict_new[rename_dict_new.values in [f]]

KeyError: False

In [160]:
{k:v for (k,v) in rename_dict_new.items() if in [f]}

SyntaxError: invalid syntax (2694754088.py, line 1)

In [ ]:
results = {item: rename_dict_new[item] for item in f}
print(results.keys())

KeyError: 'Carlos'

In [ ]:
a = pd.DataFrame({"a": np.random.rand(10), "b": np.random.rand(10), "d": np.random.rand(10)})
b = pd.DataFrame({"a": np.random.rand(10), "b": np.random.rand(10), "e": np.random.rand(10)})

common_columns = a.columns.intersection(b.columns)
a = a[common_columns]
b = b[common_columns]

In [114]:
a = a[common_columns]
b = b[common_columns]

In [ ]:
import mlflow
import os
import json

In [68]:
logged_run = mlflow.get_run("3cdd0c9d090c444fa8be1a4b7ac39882")
logged_run.to_dictionary()

{'info': {'artifact_uri': 'mlflow-artifacts:/10/3cdd0c9d090c444fa8be1a4b7ac39882/artifacts',
  'end_time': 1757067631543,
  'experiment_id': '10',
  'lifecycle_stage': 'active',
  'run_id': '3cdd0c9d090c444fa8be1a4b7ac39882',
  'run_name': 'persistent-bass-645',
  'start_time': 1757067618616,
  'status': 'FINISHED',
  'user_id': 'herrmann'},
 'data': {'metrics': {'mean squared error': 125.49499999999999,
   'root mean squared error': 11.202455088059939},
  'params': {},
  'tags': {'mlflow.user': 'herrmann',
   'mlflow.source.name': 'c:\\Users\\herrmann\\AppData\\Local\\miniforge3\\envs\\spx\\Lib\\site-packages\\ipykernel_launcher.py',
   'mlflow.source.type': 'LOCAL',
   'mlflow.runName': 'persistent-bass-645'}},
 'inputs': {'model_inputs': [],
  'dataset_inputs': [{'dataset': {'name': 'maximum water temperatures',
     'digest': 'e84aca90',
     'source_type': 'http',
     'source': '{"url": "https://mlflow.simplex4learning.de/#/experiments/10/runs/3cdd0c9d090c444fa8be1a4b7ac39882/art

In [17]:
logged_run = mlflow.get_run("3cdd0c9d090c444fa8be1a4b7ac39882")

# Use the run ID provided to get the components of the model path 
model_id = logged_run.outputs.to_dictionary()["model_outputs"][0].model_id
artifact_path = logged_run.info.artifact_uri.split("/"+logged_run.info.run_id+"/")

# Construct the artifact ID to get the model
model_path = os.path.join(artifact_path[0], "models", model_id, artifact_path[1], "model.pkl").replace("\\","/")

# Download the model and instantiate it
model = mlflow.artifacts.download_artifacts(model_path, dst_path = "./")
# loaded_model = pickle.load(open(model, 'rb'))



In [ ]:
model_input_path = os.path.join(artifact_path[0], "models", model_id, artifact_path[1], "input_example.json").replace("\\","/")
model_input = mlflow.artifacts.download_artifacts(model_input_path, dst_path = "./")

with open(model_input) as f:
    training_columns = json.load(f)["columns"]

new_columns = ["a", "Jahr", "Monat"]
target_array = []
for col in new_columns:
    target_array.append(col in training_columns)


target = new_columns[target_array.index(False)]
target

AttributeError: module 'orjson' has no attribute 'load'

In [ ]:
with open(model_input) as f:
    training_columns = json.load(f)["columns"]

In [61]:
new_columns = ["a", "Jahr", "Monat"]
target_array = []
for col in new_columns:
    target_array.append(col in training_columns)


target = new_columns[target_array.index(False)]

In [ ]:
mlflow.get_experiment(logged_run.info.experiment_id).name

'test-Gew'

In [2]:
import pandas as pd
import mlflow
import os
import pickle
import json
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, root_mean_squared_error
from mlflow.models import infer_signature

In [52]:
# Get input data from Cadenza
    # data_group = attribute_groups["input_data"]
    # input_cols = [c.name for c in data_group]
    # input_data = data[input_cols]

input_data = pd.read_parquet("../Data/water_temps.parquet")

    # Get token and set it as an environment variable
    # token= str(metadata.get_parameter("token"))
token = "eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJqNUpOUlhIX0JWZUZQS1lRanFUZ3lyWi1rYVJoUmJybGtPOWlmZjdDSDJvIn0.eyJleHAiOjE3NTc2MTY0MTEsImlhdCI6MTc1NzU4MDQxMSwianRpIjoiYWQ1ZmI0NTUtNzU5My00OWYzLTg1ZDQtMzYzZGIyYWVhYTRlIiwiaXNzIjoiaHR0cHM6Ly9hdXRoLnNpbXBsZXg0bGVhcm5pbmcuZGUvcmVhbG1zL3NpbXBsZXg0bGVhcm5pbmciLCJhdWQiOlsibWxmbG93IiwiYWNjb3VudCJdLCJzdWIiOiJjN2U2YTU2NC0xZDAyLTQxZGEtOWRlOC0xNTA2ZTc0YzM5YjUiLCJ0eXAiOiJCZWFyZXIiLCJhenAiOiJtbGZsb3ctYXBpIiwic2Vzc2lvbl9zdGF0ZSI6IjExNzBkZmNiLTZkNmQtNGE2Zi04MDJjLTVkNmI2MDhjZWI2ZSIsImFjciI6IjEiLCJhbGxvd2VkLW9yaWdpbnMiOlsiLyoiXSwicmVhbG1fYWNjZXNzIjp7InJvbGVzIjpbIm9mZmxpbmVfYWNjZXNzIiwidW1hX2F1dGhvcml6YXRpb24iLCJkZWZhdWx0LXJvbGVzLXNpbXBsZXg0bGVhcm5pbmciXX0sInJlc291cmNlX2FjY2VzcyI6eyJhY2NvdW50Ijp7InJvbGVzIjpbIm1hbmFnZS1hY2NvdW50IiwibWFuYWdlLWFjY291bnQtbGlua3MiLCJ2aWV3LXByb2ZpbGUiXX19LCJzY29wZSI6Im9wZW5pZCBwcm9maWxlIGVtYWlsIiwic2lkIjoiMTE3MGRmY2ItNmQ2ZC00YTZmLTgwMmMtNWQ2YjYwOGNlYjZlIiwiZW1haWxfdmVyaWZpZWQiOnRydWUsIm5hbWUiOiJNYXJpdXMgSGVycm1hbm4iLCJncm91cHMiOlsiQWRtaW5pc3RyYXRvciIsIkFuYWx5c3QiLCJDcmVhdG9yIiwiVmlld2VyIl0sInByZWZlcnJlZF91c2VybmFtZSI6Im1hcml1cyIsImdpdmVuX25hbWUiOiJNYXJpdXMiLCJmYW1pbHlfbmFtZSI6IkhlcnJtYW5uIiwiZW1haWwiOiJtYXJpdXMuaGVycm1hbm5AZGlzeS5uZXQifQ.Mz5aSu2z8A2NkWTACgvpNO-4fLd700TMipEY3ghb4gV7d0gxR-pe6YqPKbgSHQfGEohvhoHXSwVR69U1V-My1WsBFLEuMmERlOsLJ2zw_LzuCmOegF1LRfywqwl-ptKYEkitu120pt9l9lp8r3jj4AunKqLt2EB5d5OY1a6BSniH03IWq13GQM0RQOUQgmLR0kU7tqr0MUIiHQghKAorc3UEW0N_HtBFOicDTDalEqW1mi3FdE_ZiDzFjhd4Gh09NDl0h8nW9fhTW3B1DPf1hC67jGDfMEVdED9Fob_PZjV2uQPjd4-0UxmBkrt4k-DFQ7FbGoCZQNYM1nCKLQS6fw"
os.environ["MLFLOW_TRACKING_TOKEN"] = token


    
run_id = "3cdd0c9d090c444fa8be1a4b7ac39882" # str(metadata.get_parameter("run_id"))
logged_run = mlflow.get_run(run_id)

# Use the run ID provided to get the model name
models = mlflow.search_logged_models(
    experiment_ids=[logged_run.info.experiment_id]
)
model_name = models[models.creation_timestamp== models.creation_timestamp.max()].name

In [53]:
# Use the run ID provided to get the components of the model path 
model_id = logged_run.outputs.to_dictionary()["model_outputs"][0].model_id
model_artifact_path = logged_run.info.artifact_uri.split("/"+logged_run.info.run_id+"/")

In [54]:
# Construct the artifact ID to get the model
model_path = os.path.join(model_artifact_path[0], "models", model_id, model_artifact_path[1], "model.pkl").replace("\\","/")

# Download the model and instantiate it
model = mlflow.artifacts.download_artifacts(model_path, dst_path = "./")
loaded_model = pickle.load(open(model, 'rb'))

# Get artifact path via the dataset source logged with the dataset
artifact_path = os.path.join(logged_run.info.artifact_uri, logged_run.inputs.dataset_inputs[0].dataset.source.split("artifacts/")[-1]).replace("\\","/")[:-2]

training_data_path = mlflow.artifacts.download_artifacts(artifact_path, dst_path = "./")

training_data_old = pd.read_parquet(training_data_path)

training_data_new = pd.concat([training_data_old, input_data])

file_path = "./training_data_new.parquet"

training_data_new.to_parquet(file_path)

In [55]:
# Construct the input example path
model_input_path = os.path.join(model_artifact_path[0], "models", model_id, model_artifact_path[1], "input_example.json").replace("\\","/")
model_input = mlflow.artifacts.download_artifacts(model_input_path, dst_path = "./")

# Get the predictand
with open(model_input) as f:
    training_columns = json.load(f)["columns"]

new_columns = training_data_new.columns
target_array = []
for col in new_columns:
    target_array.append(col in training_columns)


target = new_columns[target_array.index(False)]

In [40]:
X = training_data_new.drop(target, axis=1)
temp_list = [item[0].replace(',', '.') for item in training_data_new[[target]].values]
y = np.array(temp_list, dtype=float)

print("Splitting the dataset into training and test sets...")
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=17
)

# Ensure model is trained with feature names
feature_names = training_data_new.columns.drop(target)
X_train = pd.DataFrame(X_train, columns=feature_names)
X_test = pd.DataFrame(X_test, columns=feature_names)
y_train = pd.Series(y_train, name="target")
y_test = pd.Series(y_test, name="target")

print("Defining the model hyperparameters...")
# Define the model hyperparameters
params = {
    "criterion": "squared_error",
    "random_state": 17,
}

print("Training the model...")
# Train the model
loaded_model.fit(X_train, y_train)

print("Predicting on the test set...")
# Predict on the test set
y_pred = loaded_model.predict(X_test)

print("Calculating metrics...")
# Calculate metric
mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

print("Training complete!")

print("Logging model parameters, metrics, and artifacts to MLFlow...")

Splitting the dataset into training and test sets...
Defining the model hyperparameters...
Training the model...
Predicting on the test set...
Calculating metrics...
Training complete!
Logging model parameters, metrics, and artifacts to MLFlow...


In [ ]:
with mlflow.start_run() as run:

    # Log metrics
    mlflow.log_metrics(
        metrics={"mean squared error": mse,
                "root mean squared error": rmse
                }
    )
    conda_env_filePath ="./conda_env.yml"
    if not os.path.exists(conda_env_filePath):
         print("No conda env file")
         # return ca.ErrorResponse(f"Error: Please provide a conda env file in the directory this file is run from.")
    
    # Log the model, which inherits the parameters and metric
    model_info = mlflow.sklearn.log_model(
        sk_model=loaded_model,
        conda_env=conda_env_filePath,
        # registered_model_name=model_name[0],
        signature=infer_signature(X_train, y_pred),
        input_example=X_train[:10],
        metadata={
            "author": "disy Cadenza User",
            # "version": "0.0.1",
        },
        params=params,
        tags={"Training Info": "This model was retrained via the disy Cadenza analytics extension.",
            "Additional Info": f"Predictors are {feature_names}, predictand is {target}.",
            "Extra Info": f"This model has an MSE of {mse} and an RMSE of {rmse}."
            },
        name=model_name[0],
    )

    subfolder = "data_folder"

    mlflow.log_artifact(local_path=training_data_path,
                        artifact_path=subfolder,
                        )
    
    # get the run name and id
    run_name = run.info.run_name
    run_id = run.info.run_id

    print("\nRun name: ", run_name)
    print("Run id: ", run_id)

    logged_run = mlflow.get_run(run_id)

    artifact_url_parts = run.info.artifact_uri.split('/')
    experiment_num = artifact_url_parts[1]
    artifact_path = artifact_url_parts[2]

    dataset_source_url = os.path.join("https://mlflow.simplex4learning.de/#/experiments/", experiment_num, "runs", artifact_path, "artifacts", subfolder, file_path.split("/")[-1]).replace("\\","/")

    training_dataset = mlflow.data.from_pandas(training_data_new, source = dataset_source_url, name = "maximum water temperatures", targets = target)
    mlflow.log_input(training_dataset, context="training")

    print("Complete: Everything is logged to MLFlow Server. Let's delete the variables now...")

    mlflow.end_run()

    # Delete model and training data
    os.remove(model)
    os.remove(training_data_path) 
    os.remove(file_path) 
    
model_info_frame = pd.DataFrame({"run_id":run_id, "model_id":logged_run.to_dictionary()["outputs"]["model_outputs"][0].model_id}, index = [1])


# Model id and associated run
training_metadata = [ca.ColumnMetadata(
        name="run_id",
        print_name="Run ID of new model",
        data_type=ca.DataType.STRING,
        attribute_group_name='Run ID',
        role=ca.AttributeRole.DIMENSION)],
[ca.ColumnMetadata(
        name="model_id",
        print_name="ID of new model",
        data_type=ca.DataType.STRING,
        attribute_group_name='Model ID',
        role=ca.AttributeRole.DIMENSION)]

In [67]:
temp_list[0] = float(temp_list[0])

In [68]:
temp_list[0]

21.5

In [5]:
import mlflow

In [6]:
run_id = "3cdd0c9d090c444fa8be1a4b7ac39882" # str(metadata.get_parameter("run_id"))
logged_run = mlflow.get_run(run_id)

In [9]:
logged_run

<Run: data=<RunData: metrics={'mean squared error': 125.49499999999999,
 'root mean squared error': 11.202455088059939}, params={}, tags={'mlflow.runName': 'persistent-bass-645',
 'mlflow.source.name': 'c:\\Users\\herrmann\\AppData\\Local\\miniforge3\\envs\\spx\\Lib\\site-packages\\ipykernel_launcher.py',
 'mlflow.source.type': 'LOCAL',
 'mlflow.user': 'herrmann'}>, info=<RunInfo: artifact_uri='mlflow-artifacts:/10/3cdd0c9d090c444fa8be1a4b7ac39882/artifacts', end_time=1757593334301, experiment_id='10', lifecycle_stage='active', run_id='3cdd0c9d090c444fa8be1a4b7ac39882', run_name='persistent-bass-645', start_time=1757067618616, status='FINISHED', user_id='herrmann'>, inputs=<RunInputs: dataset_inputs=[<DatasetInput: dataset=<Dataset: digest='e84aca90', name='maximum water temperatures', profile='{"num_rows": 118, "num_elements": 354}', schema=('{"mlflow_colspec": [{"type": "long", "name": "Jahr", "required": true}, '
 '{"type": "long", "name": "Monat", "required": true}, {"type": "strin